# Milestone 2: Matrix factorization

Starter notebook for milestone 2 of [Project - MovieLens Recommender](Project%20-%20MovieLens%20Recommender.md).

Cells marked **TODO** are yours to fill in; everything else runs as-is. Notebook is saved without outputs.

In [ ]:
import numpy as np, pandas as pd, pathlib, urllib.request, zipfile, io

DATA = pathlib.Path("ml-latest-small")
if not DATA.exists():
    # Or download manually from https://grouplens.org/datasets/movielens/ (ml-latest-small) and unzip here
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
ratings = pd.read_csv(DATA / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "links.csv")       # movieId, imdbId, tmdbId
ratings.head()

In [ ]:
# Leave-last-out split by time: for every user, the most recent positive is the test item,
# the one before it is validation. Positives = rating >= 4 (implicit feedback).
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]

# Contiguous ids so we can index embedding tables later
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(movies.movieId.unique()))}   # includes never-rated movies (cold items)
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

In [ ]:
def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HitRate@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (sampled protocol).
    score_fn(user_id, item_ids) -> array of scores. Switch to full ranking for final numbers,
    sampled metrics can reorder models (see Evaluating Recommenders)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum() + rng.integers(0, (s == s[0]).sum())   # 0 = first; ties broken at random (a constant scorer must score ~chance)
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

## 1. Explicit-rating MF with SGD (warm-up)
Predict $r_{ui}\approx \mu + b_u + b_i + p_u^\top q_i$ and report RMSE on a random 10% hold-out of ratings. Then move to the implicit setup below.

In [ ]:
class MF:
    def __init__(self, n_users, n_items, k=32, lr=0.01, reg=0.05, seed=0):
        rng = np.random.default_rng(seed)
        self.P = rng.normal(0, 0.1, (n_users, k)); self.Q = rng.normal(0, 0.1, (n_items, k))
        self.bu = np.zeros(n_users); self.bi = np.zeros(n_items); self.mu = 0.0
        self.lr, self.reg = lr, reg

    def predict(self, u, i):
        raise NotImplementedError  # TODO: mu + bu[u] + bi[i] + P[u] @ Q[i]

    def fit(self, u_idx, i_idx, r, epochs=20):
        raise NotImplementedError  # TODO: shuffle, loop over triples, SGD step with L2; print train RMSE per epoch

## 2. Implicit-feedback MF with ALS (TODO)
Treat training positives as 1, everything else as 0 with low confidence. Alternate closed-form updates for $P$ and $Q$. Evaluate with `evaluate`.

Check: does MF with 32 factors beat popularity? If not, check the regularisation first.

In [ ]:
# TODO: ALS (or use the `implicit` library) and score_fn = lambda u, items: Q[iid_idx(items)] @ P[uid[u]]

## 3. Sanity checks (TODO)
- Nearest neighbours of a known movie in $Q$ (cosine). Do they make sense?
- Plot HR@10 against the number of factors k and against `reg`.

In [ ]:
# TODO